# Configure and publish the TCHC data agent

Points `tchc_arrears_agent` at the **semantic model**, so questions are answered
against the same measures the report uses. Asking the lakehouse directly would let the
agent invent its own definition of "arrears rate"; asking the model means there is one
definition and everyone sees the same number.

| | |
| --- | --- |
| **Source** | `TCHC_Arrears_Vacancy` semantic model |
| **Writes** | agent instructions, table scope, published version |

The SDK surface for attaching a semantic model differs between versions, so this prints
what is available before calling it rather than assuming.

In [ ]:
AGENT_NAME = "tchc_arrears_agent"
MODEL_NAME = "TCHC_Arrears_Vacancy"

In [ ]:
import json

from fabric.dataagent.client import FabricDataAgentManagement

agent = FabricDataAgentManagement(AGENT_NAME)
print("agent:", AGENT_NAME)
print("management methods:", [n for n in dir(agent) if not n.startswith("_")])

In [ ]:
INSTRUCTIONS = """
You answer questions about Toronto Community Housing arrears and vacancy, using the
TCHC_Arrears_Vacancy semantic model. All data is synthetic and for demonstration.

Always answer with the model's measures rather than by summing columns yourself. The
measures carry definitions that have been agreed; a raw column sum will disagree with
the report and both cannot be right.

Key measures and what they mean:
- Total Arrears: the closing balance owed. It is a BALANCE, so it is not additive across
  time. Asked about a quarter or a year, report the position at the end of that period,
  never the sum of the months in it.
- Arrears Rate: share of charged households carrying any balance.
- Arrears Over 90 Days and Over 90 Day Share: the portion least likely to be recovered.
- Rent Charged, Rent Collected, Collection Rate: these are FLOWS within a period and are
  additive across time.
- Units, Units Vacant, Vacancy Rate, Revenue Forgone: vacancy position and the rent that
  represents.
- Average Turnaround Days: vacate to ready-to-rent, over COMPLETED work orders only.
  Turnarounds Open counts the unfinished ones separately.

How arrears is calculated, if asked: receipts are applied to the oldest outstanding
charge first, and the aging bucket follows the oldest charge still carrying a balance.
It is not this month's charge minus this month's payment.

Dimensions available for slicing: ward, region, tenure type (RGI or Market), unit size,
income band, building, and month.

Reporting rules:
- State the period the answer covers. If the caller did not specify one, use the latest
  period present and say which it is.
- Give units: dollars for balances, percentages for rates, days for turnaround.
- If a question cannot be answered from these measures, say so plainly rather than
  approximating. Do not estimate.
- Distinguish a measured zero from missing data. A ward with no arrears is different
  from a ward with no records.

Boundaries:
- This is management information about a portfolio, not advice about any individual
  household, and not a recommendation to take any action against a tenancy.
- Do not rank or list households for collections, enforcement or eviction, and do not
  suggest who should be contacted. Arrears affects somebody's housing, and that decision
  belongs to staff working from the full case, not to a model working from a balance.
- If asked for that, explain what you can provide instead: the shape of the portfolio,
  where balances concentrate, and how they are trending.
"""

print(INSTRUCTIONS[:400], "...")

In [ ]:
# Attach the semantic model. The method name has moved between SDK versions, so try the
# known spellings and report which one this version accepts.
attached = None
for method_name in ("add_datasource", "add_data_source"):
    method = getattr(agent, method_name, None)
    if method is None:
        continue
    import inspect
    print(f"\n{method_name}{inspect.signature(method)}")
    for kwargs in ({"name": MODEL_NAME, "type": "semantic_model"},
                   {"name": MODEL_NAME, "type": "semanticmodel"},
                   {"name": MODEL_NAME}):
        try:
            attached = method(**kwargs)
            print("  attached with", kwargs)
            break
        except Exception as error:
            print("  ", kwargs, "->", str(error).splitlines()[0][:150])
    if attached is not None:
        break

datasources = agent.get_datasources()
print("\ndatasources now:", len(datasources))
for source in datasources:
    print("  ", getattr(source, "display_name", source), type(source).__name__)

In [ ]:
# Select every table so the agent can reach the whole model, then set instructions.
for source in agent.get_datasources():
    try:
        for item in source.get_tables():
            name = item.get("display_name") if isinstance(item, dict) else item
            try:
                source.select("dbo", name)
            except TypeError:
                source.select(name)
        print("selected all tables on", getattr(source, "display_name", source))
    except Exception as error:
        print("table selection:", str(error).splitlines()[0][:160])

configuration = agent.get_configuration()
print("\nconfiguration fields:", [n for n in dir(configuration) if not n.startswith("_")])
agent.update_configuration(instructions=INSTRUCTIONS.strip())
print("instructions set")

In [ ]:
agent.publish()
print("published")

published = FabricDataAgentManagement(AGENT_NAME)
print("datasources after publish:", len(published.get_datasources()))